# 10 - KYC Risk Prediction
Train RF and XGBoost to predict KYC/AML risk.

In [1]:
import pandas as pd, numpy as np, joblib, os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, roc_curve, classification_report
import plotly.express as px, plotly.graph_objects as go
import warnings; warnings.filterwarnings('ignore')
SEED=42; np.random.seed(SEED)
PRIMARY='#635BFF'; RISK='#E74C3C'; SAFE='#27AE60'; NEUTRAL='#3498DB'; WARNING='#F39C12'; TEMPLATE='plotly_white'


In [2]:
df = pd.read_csv('data/processed/kyc_clean.csv')
print(f"Shape: {df.shape}")

# Encode sector_risk
sector_map = {'Low': 0, 'Medium': 1, 'High': 2}
df['sector_risk_encoded'] = df['sector_risk'].map(sector_map).fillna(0).astype(int)

# Fill nulls for flag columns
flag_cols = ['ofac_match_flag','fatf_txn_flag','structuring_pattern_flag',
             'rapid_movement_flag','trade_mispricing_flag','pep_flag',
             'sanctions_flag','fatf_entity_flag','ofac_country_flag',
             'sectoral_sanctions_flag','ownership_opacity_score']
for col in flag_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# Target engineering
df['kyc_risk_score'] = (
    0.25 * df['pep_flag'].fillna(0) +
    0.25 * df['sanctions_flag'].fillna(0) +
    0.15 * df['ofac_match_flag'].fillna(0) +
    0.10 * df['structuring_pattern_flag'].fillna(0) +
    0.10 * df['rapid_movement_flag'].fillna(0) +
    0.05 * df['trade_mispricing_flag'].fillna(0) +
    0.05 * df['fatf_txn_flag'].fillna(0) +
    0.05 * df['ownership_opacity_score'].fillna(0)
)
threshold = df['kyc_risk_score'].quantile(0.75)
df['kyc_risk_flag'] = (df['kyc_risk_score'] > threshold).astype(int)
print(f"Threshold: {threshold:.4f}")
print(f"Flag distribution:\n{df['kyc_risk_flag'].value_counts()}")


Shape: (50000, 23)
Threshold: 0.0250
Flag distribution:
kyc_risk_flag
0    40167
1     9833
Name: count, dtype: int64


In [3]:
# Features and split
feature_cols = flag_cols + ['sector_risk_encoded']
feature_cols = [c for c in feature_cols if c in df.columns]
X = df[feature_cols]
y = df['kyc_risk_flag']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
ratio = (y_train==0).sum() / (y_train==1).sum()
print(f"Scale pos weight: {ratio:.2f}")


Train: (40000, 12), Test: (10000, 12)
Scale pos weight: 4.09


In [4]:
# Train models
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)

xgb = XGBClassifier(scale_pos_weight=ratio, eval_metric='auc', random_state=SEED, use_label_encoder=False)
xgb.fit(X_train, y_train)

for name, model in [('Random Forest', rf), ('XGBoost', xgb)]:
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:,1]
    print(f"\n{name}:")
    print(f"  Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"  F1: {f1_score(y_test, y_pred):.4f}")
    print(f"  ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")
    print(classification_report(y_test, y_pred))



Random Forest:
  Accuracy: 1.0000
  F1: 1.0000
  ROC-AUC: 1.0000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      8033
           1       1.00      1.00      1.00      1967

    accuracy                           1.00     10000
   macro avg       1.00      1.00      1.00     10000
weighted avg       1.00      1.00      1.00     10000


XGBoost:
  Accuracy: 1.0000
  F1: 1.0000
  ROC-AUC: 1.0000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      8033
           1       1.00      1.00      1.00      1967

    accuracy                           1.00     10000
   macro avg       1.00      1.00      1.00     10000
weighted avg       1.00      1.00      1.00     10000



In [5]:
# Assign risk_level using best model (XGBoost)
best_model = xgb
proba = best_model.predict_proba(X)[:,1]
def risk_level(p):
    """Assign risk level from probability."""
    if p > 0.75: return 'Critical'
    elif p > 0.50: return 'High Risk'
    elif p > 0.25: return 'Medium Risk'
    else: return 'Low Risk'
df['risk_level'] = pd.Series(proba).apply(risk_level)
print("Risk level distribution:")
print(df['risk_level'].value_counts())


Risk level distribution:


risk_level
Low Risk    40167
Critical     9833
Name: count, dtype: int64


In [6]:
# Plot a: risk_level donut
rl = df['risk_level'].value_counts().reset_index(); rl.columns=['level','count']
fig = px.pie(rl, values='count', names='level', hole=0.4,
             color='level', color_discrete_map={'Critical':RISK,'High Risk':WARNING,'Medium Risk':NEUTRAL,'Low Risk':SAFE},
             template=TEMPLATE, title='KYC Risk Level Distribution')
fig.show()


In [7]:
# Plot b: sector_risk vs kyc_risk_flag
ct = df.groupby(['sector_risk','kyc_risk_flag']).size().reset_index(name='count')
ct['kyc_risk_flag'] = ct['kyc_risk_flag'].map({0:'Low Risk',1:'High Risk'})
fig = px.bar(ct, x='sector_risk', y='count', color='kyc_risk_flag', barmode='group',
             color_discrete_map={'Low Risk':SAFE,'High Risk':RISK},
             template=TEMPLATE, title='Sector Risk vs KYC Risk Flag')
fig.show()


In [8]:
# Plot c: pep_flag vs sanctions_flag
ct = df.groupby(['pep_flag','sanctions_flag']).size().reset_index(name='count')
ct['pep_flag'] = ct['pep_flag'].astype(str)
ct['sanctions_flag'] = ct['sanctions_flag'].astype(str)
fig = px.bar(ct, x='pep_flag', y='count', color='sanctions_flag', barmode='stack',
             template=TEMPLATE, title='PEP Flag vs Sanctions Flag Co-occurrence')
fig.show()


In [9]:
# Plot d-e: structuring + opacity
v = df['structuring_pattern_flag'].value_counts().reset_index(); v.columns=['flag','count']
fig = px.bar(v, x='flag', y='count', color_discrete_sequence=[NEUTRAL], template=TEMPLATE, title='Structuring Pattern Flag')
fig.show()

fig = px.box(df.dropna(subset=['ownership_opacity_score']), x='risk_level', y='ownership_opacity_score',
             color='risk_level', color_discrete_map={'Critical':RISK,'High Risk':WARNING,'Medium Risk':NEUTRAL,'Low Risk':SAFE},
             template=TEMPLATE, title='Ownership Opacity Score by Risk Level')
fig.show()


In [10]:
# Plot f: ROC curves
fig = go.Figure()
for name, model, color in [('Random Forest', rf, PRIMARY), ('XGBoost', xgb, RISK)]:
    y_proba = model.predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, name=f"{name} (AUC={auc:.3f})", line=dict(color=color)))
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], name='Random', line=dict(dash='dash', color='gray')))
fig.update_layout(title='ROC Curves', xaxis_title='FPR', yaxis_title='TPR', template=TEMPLATE)
fig.show()


In [11]:
# Plot g: Feature importance RF
imp = pd.DataFrame({'feature': feature_cols, 'importance': rf.feature_importances_})
imp = imp.sort_values('importance', ascending=True)
fig = px.bar(imp, y='feature', x='importance', orientation='h',
             color_discrete_sequence=[PRIMARY], template=TEMPLATE, title='Feature Importance - Random Forest')
fig.show()


In [12]:
# Plot h: Top 15 client_country by high risk
if 'client_country' in df.columns:
    hr_country = df[df['kyc_risk_flag']==1]['client_country'].value_counts().head(15).reset_index()
    hr_country.columns = ['country','count']
    fig = px.bar(hr_country, x='country', y='count', color_discrete_sequence=[RISK],
                 template=TEMPLATE, title='Top 15 Client Countries by High Risk Count')
    fig.update_xaxes(tickangle=45)
    fig.show()


In [13]:
# Save
os.makedirs('models/kyc', exist_ok=True)
joblib.dump(rf, 'models/kyc/random_forest_kyc.pkl')
joblib.dump(xgb, 'models/kyc/xgboost_kyc.pkl')
print("Saved KYC models")

save_cols = ['client_id','kyc_risk_score','risk_level'] + feature_cols
df[save_cols].to_parquet('data/features/kyc_features.parquet', index=False)
print("Saved kyc_features.parquet")


Saved KYC models


Saved kyc_features.parquet
